# 08. Lakeflow Declarative Pipelines - 手順ではなく結果を宣言する

`01`〜`07` では、処理を **手順** で書いてきました。
「このファイルを読んで、こう変換して、このテーブルに書く」を自分で並べ、
チェックポイントの置き場所を決め、実行する順番も自分で決めていました。

Lakeflow Declarative Pipelines (LDP) は、これを **結果の宣言** に置き換えます。
「このテーブルはこういう中身である」とだけ書けば、
読む順番・増分の管理・チェックポイントの置き場所はエンジンが決めます。

以前は Delta Live Tables (DLT) と呼ばれていたもので、ネット上の記事の多くは `import dlt` で書かれています。
現在の書き方は `from pyspark import pipelines as dp` です。

このノートブックで確かめること:

1. ストリーミングテーブルとマテリアライズドビューの違い
2. 実行順をどこにも書いていないのに、正しい順で動くこと
3. もう一度流したとき、それぞれがどう振る舞うか
4. いつLDPを選ぶか

**前提**: `00_setup` を実行済みであること。`01` `04` を読んでいること。


## 準備


In [1]:
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.runtime import dbutils

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

# パイプラインの起動はSparkではなくREST API経由になる。その入口が WorkspaceClient
w = WorkspaceClient(profile="free")

In [2]:
CATALOG = "tech_survey"

# パイプラインの出力先。他のトピックのテーブルと混ざらないよう、00_setup で分けてある
SCHEMA = "ldp"
LANDING = f"/Volumes/{CATALOG}/ops/landing/08_ldp"

# resources/pipelines/08_medallion.pipeline.yml で付けた名前
PIPELINE_NAME = "08_medallion_ldp"

## 1. 取り込み元を用意する

パイプラインが読むJSONを置きます。ここは `01` と同じやり方です。


In [3]:
import json
import random
import uuid

# データを格納するためのディレクトリを削除（存在する場合）
try:
    dbutils.fs.rm(LANDING, True)
except NotFound:
    pass


# 後でもう一度足すので、まとめておく
def put_orders(name: str, n: int) -> None:
    """ランダムな注文データを作る
    
    Parameters
    ----------
    name : str
        ファイル名のベース。`{name}.json` というファイルが LANDING に作られる
    n : int
        作る行数
    """
    rows = [
        {
            "order_id": str(uuid.uuid4()),
            "product": random.choice(["laptop", "monitor", "keyboard"]),
            "amount": random.randint(1000, 50000),
            "event_time": "2026-09-13T10:00:00",
        }
        for _ in range(n)
    ]
    dbutils.fs.put(f"{LANDING}/{name}.json", "\n".join(json.dumps(r) for r in rows), True)


put_orders("orders_1", 20)
dbutils.fs.ls(LANDING)

[FileInfo(path='/Volumes/tech_survey/ops/landing/08_ldp/orders_1.json', name='orders_1.json', size=2571, modificationTime=1789299125000)]

## 2. パイプラインのソースを読む

ソースは [`src/pipelines/08_medallion/transformations/`](../../pipelines/08_medallion/transformations) に3つ置いてあります。

**bronze_orders.py**

```python
from pyspark import pipelines as dp

LANDING = "/Volumes/tech_survey/ops/landing/08_ldp"


# readStream で読むと「ストリーミングテーブル」になる。届いたぶんだけ追記される
@dp.table(comment="landing に届いた注文JSONをそのまま取り込む")
def bronze_orders():
    return spark.readStream.format("cloudFiles").option("cloudFiles.format", "json").load(LANDING)
```

**silver_orders.py**

```python
@dp.table(comment="型を整えて、金額が正の注文だけを残す")
def silver_orders():
    return (
        spark.readStream.table("bronze_orders")   # 同じパイプライン内のテーブルは名前だけで参照する
        .withColumn("event_time", F.col("event_time").cast("timestamp"))
        .withColumn("amount", F.col("amount").cast("int"))
        .filter(F.col("amount") > 0)
    )
```

**gold_sales.py**

```python
@dp.table(comment="商品ごとの売上合計")
def gold_sales():
    return (
        spark.read.table("silver_orders")   # readStream ではなく read
        .groupBy("product")
        .agg(
            F.count("*").alias("order_count"),
            F.sum("amount").alias("total_amount"),
        )
    )
```

注目してほしいのが4つあります。

1. **`spark` を作っていない。** パイプラインの中では最初から使える
2. **`read` か `readStream` か** で、できるテーブルの種類が変わる
3. **チェックポイントの指定が無い。** `01` では自分で場所を決めて渡していた
4. **ファイル名に順番が付いていない。** 付ける必要が無いから (理由は後で確かめます)


## 3. デプロイする

パイプラインは「セルを実行して動かす」ものではありません。
ソースをワークスペースに配置し、パイプラインとして登録する必要があります。それがデプロイです。

リポジトリのルートで、ターミナルから実行してください。

```sh
databricks bundle deploy
```

これで2つのことが行われます。

- `src/` 以下のファイルがワークスペースに同期される
- `resources/pipelines/08_medallion.pipeline.yml` の内容でパイプラインが作られる

`-t dev` のようにターゲットを指定している例をよく見ますが、ここでは要りません。
`databricks.yml` で `dev` に `default: true` を付けてあるので、省略するとこれが選ばれます。

ターゲットが1つのうちは省略できますが、`prod` を足したら
**省略すると意図しないほうへデプロイされる** ことになります。その時点から明示するのが安全です。

**ソースを書き換えたときは、また `deploy` が必要です。**
一方、データが増えただけなら deploy は要りません。起動は次のセルからできます。


In [4]:
# バンドルは dev ターゲットで名前に接頭辞を付けるので、後方一致で探す
PIPELINE_ID = next(
    p.pipeline_id for p in w.pipelines.list_pipelines() if p.name.endswith(PIPELINE_NAME)
)

# この画面を開くと、処理の流れ (DAG) と各テーブルの行数が見える
print(f"{w.config.host}/pipelines/{PIPELINE_ID}")

https://dbc-6f498009-072a.cloud.databricks.com/pipelines/4712986a-5d50-497d-be47-9842a69a0563


In [5]:
import time

from databricks.sdk.service.pipelines import UpdateInfoState

# 更新の終わりを表す状態
DONE = (UpdateInfoState.COMPLETED, UpdateInfoState.FAILED, UpdateInfoState.CANCELED)


# 何度か起動するので、まとめておく
def run_pipeline(full_refresh: bool = False) -> UpdateInfoState:
    """パイプラインを起動し、完了するまで待つ。

    Parameters
    ----------
    full_refresh : bool
        True の場合、フルリフレッシュで起動する

    Returns
    -------
    UpdateInfoState
        パイプラインの更新状態
    """
    update = w.pipelines.start_update(pipeline_id=PIPELINE_ID, full_refresh=full_refresh)

    # 終わるまで待つ。サーバーレスでも最初の起動には数分かかる
    while True:
        state = w.pipelines.get_update(PIPELINE_ID, update.update_id).update.state
        print(state.value)
        if state in DONE:
            return state
        time.sleep(20)

In [6]:
# 1回目は full_refresh で、まっさらな状態から作る
run_pipeline(full_refresh=True)

CREATED
WAITING_FOR_RESOURCES
INITIALIZING
SETTING_UP_TABLES
RUNNING
COMPLETED


<UpdateInfoState.COMPLETED: 'COMPLETED'>

In [7]:
# 3つのテーブルができているはず。それぞれ何行になったかを見る
for t in ("bronze_orders", "silver_orders", "gold_sales"):
    print(t, spark.table(f"{CATALOG}.{SCHEMA}.{t}").count())

display(spark.table(f"{CATALOG}.{SCHEMA}.gold_sales"))

bronze_orders 20
silver_orders 20
gold_sales 3


,product,order_count,total_amount
0,laptop,8,197641
1,keyboard,7,160303
2,monitor,5,62784


bronze → silver → gold の順で処理されました。
**その順番は、どのファイルにも書いていません。**

エンジンは各関数が読んでいるテーブル名を見て、依存関係を組み立てています。
`gold_sales` は `silver_orders` を読む、`silver_orders` は `bronze_orders` を読む、と書いてあるので、
順番はそこから決まります。ファイルを3つに分けても、名前を付け替えても関係ありません。

`01`〜`07` では、この順番を自分で並べていました。
テーブルが増えるほど、並べ間違いと「変更したら別のところが壊れる」が起きやすくなる部分です。


In [8]:
# 同じ @dp.table で作ったのに、テーブルの種類が違うことを確認する
display(
    spark.sql(f"""
        SELECT table_name, table_type
        FROM {CATALOG}.information_schema.tables
        WHERE table_schema = '{SCHEMA}'
        ORDER BY table_name
    """)
)

,table_name,table_type
0,__materialization_mat_4712986a_5d50_497d_be47_9842a69a0563_bronze_orders_1,MANAGED
1,__materialization_mat_4712986a_5d50_497d_be47_9842a69a0563_gold_sales_1,MANAGED
2,__materialization_mat_4712986a_5d50_497d_be47_9842a69a0563_silver_orders_1,MANAGED
3,bronze_orders,STREAMING_TABLE
4,event_log_4712986a_5d50_497d_be47_9842a69a0563,MANAGED
5,gold_sales,MATERIALIZED_VIEW
6,silver_orders,STREAMING_TABLE


`bronze_orders` と `silver_orders` は **ストリーミングテーブル**、
`gold_sales` は **マテリアライズドビュー** になっているはずです。

違いは中で `readStream` を使ったか `read` を使ったかだけでした。

| | ストリーミングテーブル | マテリアライズドビュー |
|---|---|---|
| 何をするか | 届いたぶんだけ追記する | 毎回まとめて計算し直す |
| 元データの変更 | 追記しか想定しない | 過去が変わっても正しくなる |
| 向いている層 | Bronze / Silver | Gold (集計) |

集計は、過去の行が1つ変わると結果も変わります。
毎回計算し直すマテリアライズドビューのほうが素直、というわけです。


## 4. もう一度流す

データを10件足して、`full_refresh` を付けずに起動します。
ストリーミングテーブルとマテリアライズドビューで、何が起きるか予想してみてください。


In [9]:
# データを10件追加
put_orders("orders_2", 10)

# 2回目は full_refresh ではなく、前回の状態を引き継いで追加分だけ処理する
run_pipeline(full_refresh=False)

# 各テーブルの行数を表示
for t in ("bronze_orders", "silver_orders", "gold_sales"):
    print(t, spark.table(f"{CATALOG}.{SCHEMA}.{t}").count())

CREATED
WAITING_FOR_RESOURCES
INITIALIZING
RUNNING
RUNNING
COMPLETED
bronze_orders 30
silver_orders 30
gold_sales 3


どれも増えた数だけ結果に反映されています。ただし、やっていることは違います。

- **ストリーミングテーブル** は、追加された10件だけを処理して追記しました。
  最初の20件は読み直していません。`01` のチェックポイントと同じ仕組みですが、**置き場所を決めたのはエンジン** です
- **マテリアライズドビュー** は、30件を対象に集計をやり直しました

処理した行数はパイプラインの画面 (3章で出したURL) で見えます。
各テーブルの箱をクリックすると、その更新で何行流れたかが出ます。

`full_refresh=True` を付けると、ストリーミングテーブルも最初から読み直します。
取り込みロジックを変えたときなど、作り直したい場合に使います。

ただし **本番では慎重に使うものです。** ストリーミングの状態を捨てて最初からやり直すので、
取り込み元がもう無い場合 (ファイルを消した、Kafkaの保持期間を過ぎた等) は
**復元できない行が出ます。**
ここでは作ったばかりのパイプラインなので、安全に使えています。


### `04` の `replaceWhere` はどこへ行ったか

`04` の `replaceWhere` は、**再実行したときに特定の範囲だけを置き換える** 手段でした。
「9月11日のデータが間違っていたので、その日の分だけ作り直す」という場面です。

LDPでは、やり直しの話が3つに分かれます。

**1. マテリアライズドビュー (gold) は、やり直しの指定が要りません。**
「silver を集計したもの」と宣言してあるだけなので、silver が直れば次の更新で結果も直ります。

**2. 普通のストリーミングテーブル (bronze / silver) は、過去を直せません。**
追記しかしない前提で、どのファイルを処理済みかをエンジンが覚えているので、
元のファイルを差し替えても読み直しません。直すなら `full_refresh` = 全部作り直すことになります。

**3. そして、範囲を限定してやり直すための仕組みもあります。REPLACE WHERE フローです。**

```python
@dp.table(replace_where=F.col("order_date") >= F.date_sub(F.current_date(), 7))
def orders_enriched():
    return spark.read.table("orders_fct").join(spark.read.table("product_dim"), "product_id")
```

`replace_where` の条件に当たる行を消して、同じ条件でソースを計算し直して入れ直します。
条件の外は触りません。やっていることは `04` と同じです。

違うのは **どこに書くか** です。
`04` では書き込むときのオプションでした。「今回はこの範囲を上書きする」という手続きです。
LDPでは **テーブル定義の一部** になります。「このテーブルは常にこの範囲を作り直す」という宣言です。

一度だけ別の範囲を作り直したいとき (過去分のバックフィルなど) は、
更新を起動するときに条件を上書きできます。これが `replace_where_overrides` です。
定義を書き換えずに、その回だけ範囲を変えられます。

制約もあります。

- 1つのテーブルに REPLACE WHERE フローは1つだけ
- AUTO CDC や append flow と併用できない
- **expectations (`09` で扱うデータ品質チェック) が使えない**
- 増分リフレッシュにはサーバーレスが必要。条件を満たさないと毎回まるごと計算し直す

参考: [Batch processing with REPLACE WHERE flows](https://docs.databricks.com/aws/en/ldp/flows-replace-where)


## 5. いつLDPを選ぶか

| | 自分で書く (01〜07) | LDP |
|---|---|---|
| 実行順 | 自分で並べる | 依存関係から自動で決まる |
| チェックポイント | 置き場所を決めて管理する | エンジンが持つ |
| 増分処理 | 自分で設計する | 宣言すれば任される |
| 作り直し | 自分で消して作り直す | `full_refresh` で足りる |
| 細かい制御 | 何でもできる | 枠の中でしかできない |
| 動かし方 | ノートブック / ジョブ | パイプラインとしてデプロイする |

**素直な medallion 構成なら、LDPのほうが書く量が減ります。**
順番もチェックポイントも考えなくてよくなるぶん、変換ロジックだけが残ります。

選びにくいのは、枠から外れることをしたいときです。

- テーブル以外への書き込み (外部DB、API呼び出しなど)
- 1回の処理の中で細かく条件分岐したい
- 既存のジョブやスケジュールの一部に組み込みたい

`06` の `foreachBatch` のように **書き込みを自分で制御したい話とは相性が悪い**、と考えると分かりやすいです。
あちらは制御を取りに行く方向、LDPは制御を手放す方向です。


## 考えてみる

- `bronze_orders` を `spark.read`  (readStream でない) に変えると、何が変わりますか
- LDPでも `06` でやった冪等性の心配は要るのでしょうか
- ソースを書き換えたのに `deploy` を忘れて起動すると、どうなるでしょうか


### 答え

**Q1. `read` に変えると**

ストリーミングテーブルではなく、マテリアライズドビューになります。
起動のたびに `LANDING` のファイルを **全部読み直して** 作り直すことになります。

少量なら問題ありませんが、ファイルが溜まるほど毎回の処理が重くなります。
Bronze は増える一方の層なので、増分で済むストリーミングテーブルが向いています。

**Q2. 冪等性の心配は要るか**

**ほぼ要らなくなります。** ここがLDPの効きどころです。

`06` で `txnVersion` が必要だったのは、`foreachBatch` で **自分で書き込んでいた** からでした。
LDPでは書き込み自体をエンジンに任せているので、
更新が途中で落ちても、やり直しで二重にならないように向こうが面倒を見ます。

`06` の表現を借りると、`foreachBatch` を選ぶのが「冪等性を自分で引き受ける」ことなら、
LDPを選ぶのは「引き受けない」ことです。

**Q3. deploy を忘れると**

**古いソースのまま動きます。** エラーにはなりません。

パイプラインが読むのはワークスペースに同期されたファイルであって、
手元のファイルではないからです。
「直したはずなのに結果が変わらない」ときは、まず `deploy` を疑うことになります。


## 後片付け

このノートブックで作ったものを消したいときだけ、コメントを外して実行します。
パイプライン自体は `databricks bundle destroy` で消せます。


In [ ]:
# for t in ("gold_sales", "silver_orders", "bronze_orders"):
#     spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.{t}")
# dbutils.fs.rm(LANDING, True)